# Рекуррентные нейронные сети для NLP

К концу 2000-х языковое моделирование опиралось на n-граммы со сглаживанием, а нейросетевая языковая модель с фиксированным окном [(Bengio et al., 2003)](https://jmlr.org/papers/v3/bengio03a.html) уже показала силу распределённых представлений слов. Обе конструкции ограничены: контекст обрезан заранее заданным окном. **Рекуррентные нейронные сети** (recurrent neural networks, RNN) [(Elman, 1990)](https://doi.org/10.1207/s15516709cog1402_1) сняли это ограничение: состояние фиксированного размера, обновляемое на каждом шаге, в принципе способно нести информацию о префиксе любой длины. Рекуррентная языковая модель [(Mikolov et al., 2010)](https://doi.org/10.21437/Interspeech.2010-343) первой уверенно обошла n-граммы, и примерно на десятилетие (2010–2017) RNN стали доминирующей архитектурой NLP. Эта глава прослеживает всю арку: от простой рекуррентной ячейки через борьбу с затухающими градиентами (LSTM, GRU) к encoder–decoder и вниманию — механизму, из которого вырастет Transformer, — и к современному возвращению рекуррентности в лице state-space моделей.

## Введение в обработку последовательностей

### Отказ от гипотезы i.i.d.

Классическое обучение с учителем предполагает выборку независимых одинаково распределённых пар $(x_i, y_i)$ фиксированной размерности. Текст нарушает оба допущения: токены внутри последовательности зависимы (порядок несёт смысл), а длина переменная. Вероятностная постановка задаётся цепным правилом

$$p(x_1,\dots,x_T)=\prod_{t=1}^{T}p(x_t\mid x_1,\dots,x_{t-1}),$$

то есть моделировать нужно условные распределения с неограниченно растущим контекстом. N-граммы делают марковское допущение порядка $n-1$, нейросетевая LM Бенджио обрезает контекст фиксированным окном; нужен механизм, сворачивающий префикс произвольной длины в представление постоянного размера.

### Типы задач по форме входа и выхода

По соотношению входа и выхода выделяют четыре режима: many-to-one — последовательность в один выход (классификация тональности, детекция спама); one-to-many — генерация последовательности из одного входа (порождение текста по затравке, описание изображения); синхронный many-to-many — выход на каждом шаге, $T_{вх}=T_{вых}$ (частеречная разметка, NER); асинхронный many-to-many — длины входа и выхода различаются (перевод, суммаризация), что потребует отдельной архитектуры encoder–decoder. Все четыре режима обслуживаются одной и той же рекуррентной ячейкой — меняется только то, где снимается выход.

### Полносвязные сети против рекуррентных

Полносвязная сеть требует вход фиксированной размерности: текст приходится обрезать или скользить окном, число параметров растёт с шириной окна, а паттерн, выученный в позиции 5, не переносится в позицию 50. RNN применяет одни и те же матрицы на каждом шаге — разделение весов во времени, прямой аналог разделения весов свёртки по пространству. Следствия: число параметров не зависит от $T$; инвариантность к позиции паттерна; последовательность любой длины обрабатывается за один проход.

### Скрытое состояние как память

Информация о префиксе аккумулируется в векторе $h_t\in\mathbb{R}^{d}$ — скрытом состоянии. Модель делает марковское допущение на уровне состояния: $p(x_{t+1}\mid x_{\le t})\approx p(x_{t+1}\mid h_t)$, то есть $h_t$ — обучаемая достаточная статистика префикса. Это сжатие с потерями: сколь угодно длинная история упаковывается в $d$ чисел. Отсюда и сила RNN (константная память на инференсе), и её главный будущий конфликт — информационное бутылочное горлышко, которое проявится в seq2seq.

## Математический аппарат классической RNN

### Уравнения состояния и выхода

Сеть Элмана:

$$h_t=f\big(W_{xh}\,x_t+W_{hh}\,h_{t-1}+b_h\big),\qquad y_t=g\big(W_{hy}\,h_t+b_y\big),$$

где $x_t\in\mathbb{R}^{e}$ — вход шага (обычно эмбеддинг токена), $W_{xh}\in\mathbb{R}^{d\times e}$, $W_{hh}\in\mathbb{R}^{d\times d}$, $f$ — поэлементная нелинейность (как правило tanh), $g$ — под задачу: softmax для языковой модели, сигмоида для бинарной классификации. Прошлое и настоящее встречаются в единственном месте — сумме $W_{hh}h_{t-1}+W_{xh}x_t$.

### Развёртывание во времени

Подстановка рекурсии в саму себя разворачивает (unfolding) RNN в граф вычислений глубины $T$ с общими весами: RNN на последовательности из 100 токенов — это стослойная сеть, все слои которой разделяют одну матрицу $W_{hh}$. Отсюда выразительность (композиция $T$ нелинейностей) — и все патологии обучения очень глубоких сетей, усиленные повторным умножением на одну и ту же матрицу.

### Выбор функции активации

tanh ограничен интервалом $(-1,1)$ — состояние не может неограниченно расти — и ноль-центрирован; плата — $|\tanh'|\le 1$ и насыщение на хвостах, питающие затухание градиента. ReLU не насыщается и держит градиент 1 на активной части, но рекуррентное применение неограниченной функции взрывает активации при $\sigma_1(W_{hh})>1$. Компромисс продемонстрировала **IRNN** [(Le et al., 2015)](https://arxiv.org/abs/1504.00941): ReLU-сеть с инициализацией $W_{hh}=I$, $b=0$ на старте просто копирует состояние и на длинных зависимостях сопоставима с LSTM. Дефолт классических RNN — tanh.

### Инициализация начального состояния

$h_0=0$ — стандарт: «до начала текста контекста нет». Обучаемый $h_0$ полезен на коротких последовательностях, где старт вносит заметный вклад: выучивается априорный контекст. Случайный шум в $h_0$ при обучении — лёгкая регуляризация, снижающая зависимость от начала. Stateful-режим — $h_0$ очередного чанка равен $h_T$ предыдущего — используется при обучении на длинных потоках усечённым BPTT (см. ниже).

## Обратное распространение ошибки во времени

### Вывод градиента

**BPTT** (backpropagation through time) [(Werbos, 1990)](https://doi.org/10.1109/5.58337) — обычный backprop, применённый к развёрнутому графу. Для суммарных потерь $L=\sum_t L_t$ вклад шага $t$ в градиент по $W_{hh}$ собирается со всех предшествующих позиций $k\le t$:

$$\frac{\partial L_t}{\partial W_{hh}}=\sum_{k=1}^{t}\frac{\partial L_t}{\partial h_t}\left(\prod_{i=k+1}^{t}\frac{\partial h_i}{\partial h_{i-1}}\right)\frac{\partial^{+} h_k}{\partial W_{hh}},$$

где $\partial^{+}h_k/\partial W_{hh}$ — «немедленная» производная при замороженном $h_{k-1}$. Вся динамика обучения спрятана в произведении якобианов соседних шагов.

### Произведение якобианов

Для $h_i=f(a_i)$, $a_i=W_{hh}h_{i-1}+W_{xh}x_i+b_h$ якобиан одного шага

$$\frac{\partial h_i}{\partial h_{i-1}}=\operatorname{diag}\big(f'(a_i)\big)\,W_{hh}.$$

Градиент от потерь на шаге $t$ до состояния шага $k$ проходит через $t-k$ таких множителей и ведёт себя как степень матрицы: норма произведения ограничена

$$\Big\lVert\prod_{i=k+1}^{t}\frac{\partial h_i}{\partial h_{i-1}}\Big\rVert\le\big(\gamma\,\sigma_1(W_{hh})\big)^{\,t-k},$$

где $\sigma_1$ — наибольшее сингулярное число, а $\gamma=\sup|f'|$: 1 для tanh, 1/4 для сигмоиды.

### Затухающие градиенты

При $\sigma_1<1/\gamma$ вклад далёких шагов затухает экспоненциально — это достаточное условие. В линейном приближении судьбу решает спектральный радиус: $\rho(W_{hh})<1$ — затухание, $\rho>1$ — возможен взрыв. Практическое следствие: vanilla RNN надёжно выучивает зависимости на 5–20 шагов, а сигнал вида «слово в начале абзаца определяет форму глагола в конце» до весов просто не доходит. Диагноз поставлен в работах [(Bengio et al., 1994)](https://doi.org/10.1109/72.279181) и в дипломной работе Хохрайтера (1991); строгие условия и геометрическая картина — в [(Pascanu et al., 2013)](https://arxiv.org/abs/1211.5063).

### Взрывающиеся градиенты

Симметричная патология: при $\gamma\,\sigma_1>1$ норма может расти экспоненциально (это необходимое условие взрыва). Взрывы редки, но катастрофичны: один гигантский шаг SGD выбрасывает параметры из выученного бассейна — спайки лосса, NaN. Геометрически это «стены» на ландшафте потерь вблизи мест, где рекуррентная динамика меняет режим.

### Клиппинг градиентов

Клиппинг по норме: если $\lVert g\rVert>\theta$, то $g\leftarrow\theta\,g/\lVert g\rVert$ — длина ограничивается, направление сохраняется; типичные $\theta\in[1,5]$. Клиппинг по значению: $g_i\leftarrow\operatorname{clip}(g_i,-\theta,\theta)$ — проще, но искажает направление. Клиппинг — стандартная страховка любой рекуррентной модели, однако лечит он только взрыв; затухание требует архитектурного решения, и им станет LSTM.

## Подготовка данных и пайплайн обучения

### Батчи переменной длины

Батч — прямоугольный тензор $B\times T_{\max}$: короткие последовательности дополняются PAD-токеном, а маска исключает паддинг из функции потерь и из агрегаций (пулинга). Bucketing — группировка примеров близкой длины перед батчированием — сокращает долю паддинга в разы ценой лёгкого нарушения случайности перемешивания.

### Packing и unpacking

Чтобы не тратить вычисления на паддинг, фреймворки переупаковывают батч: в PyTorch `pack_padded_sequence` строит плоский буфер валидных шагов со списком `batch_sizes` (сколько последовательностей ещё «живы» на каждом шаге $t$), cuDNN исполняет ровно нужное, `pad_packed_sequence` возвращает прямоугольную форму. Для двунаправленных сетей packing практически обязателен: без него обратный проход стартует с PAD-токенов, а не с реального конца последовательности.

### Онлайн- и батчевое обучение

Онлайн-режим ($B=1$): минимальные задержка и память, естественен для потоков, но градиент шумный и параллелизм GPU не используется. Мини-батчи — стандарт: усреднённый градиент и матричные операции. Ключевая асимметрия RNN: вычисления параллелятся по батчу $B$, но не по времени $T$ — каждый шаг ждёт предыдущего. Это ограничение станет центральным пунктом критики в конце главы.

### Truncated BPTT

**Truncated BPTT** [(Williams & Peng, 1990)](https://doi.org/10.1162/neco.1990.2.4.490): длинный поток режется на чанки по $k$ шагов; состояние переносится между чанками вперёд (с отсоединением от графа — detach), а backprop идёт только внутри чанка. Память падает с $O(Td)$ до $O(kd)$ активаций, обновления учащаются. Цена — смещение градиента: через границы чанков он не течёт, и зависимости длиннее $k$ напрямую не обучаются, лишь косвенно через перенесённое состояние. Общая форма TBPTT($k_1$, $k_2$) — обновление каждые $k_1$ шагов с разворачиванием на $k_2$ назад.

### Авторегрессивный инференс

Генерация — цикл: предсказанный $\hat y_t$ подаётся на вход шага $t+1$ (жадный argmax, сэмплирование или beam search — детали в кейсах). Состояние обновляется на месте, поэтому RNN генерирует каждый следующий токен за $O(1)$ памяти и вычислений независимо от длины уже сгенерированного. Этого свойства будут лишены трансформеры (KV-кэш растёт линейно), и его же вернут state-space модели.

## Инициализация и регуляризация

### Ортогональная инициализация

**Ортогональная инициализация** [(Saxe et al., 2013)](https://arxiv.org/abs/1312.6120): $W_{hh}$ берётся ортогональной (QR-разложение случайной гауссовской матрицы). Все сингулярные числа равны 1, линейная часть рекурсии изометрична: нормы сигнала и градиента на старте не растут и не затухают — произведения якобианов ведут себя смирно ровно тогда, когда это важнее всего, в начале обучения.

### L1/L2-регуляризация рекуррентных весов

Weight decay стягивает веса к нулю, то есть уменьшает $\sigma_1(W_{hh})$ — и тем самым усугубляет затухание градиентов. Практика: исключать $W_{hh}$ из weight decay или ставить для неё коэффициент на порядок меньше, а полновесный L2 применять к входным и выходным матрицам. L1 (разреживание) на рекуррентных весах используется редко.

### Вариационный dropout

Наивный dropout с новой маской на каждом шаге умножает память на цепочку случайных масок — сигнал деградирует как $(1-p)^{T}$. Первый практичный рецепт [(Zaremba et al., 2014)](https://arxiv.org/abs/1409.2329) — дропаут только на нерекуррентных связях, между слоями стека. **Вариационный dropout** [(Gal & Ghahramani, 2016)](https://arxiv.org/abs/1512.05287) идёт дальше: одна и та же маска на все шаги последовательности — на входах, выходах, рекуррентных связях и эмбеддингах; обоснование — интерпретация dropout как байесовского вариационного вывода.

### Zoneout

**Zoneout** [(Krueger et al., 2016)](https://arxiv.org/abs/1606.01305): вместо зануления активаций — стохастический отказ от обновления,

$$h_t=m_t\odot h_{t-1}+(1-m_t)\odot\tilde h_t,\qquad m_t\sim\operatorname{Bernoulli}(p)$$

покомпонентно. Информация не уничтожается, а замораживается; тождественная связь через замороженные компоненты дополнительно улучшает поток градиента сквозь время. На инференсе используется детерминированное среднее.

### Layer Normalization

**Layer Normalization** [(Ba et al., 2016)](https://arxiv.org/abs/1607.06450) нормирует преактивации по признакам в пределах одного примера и одного шага:

$$\operatorname{LN}(a)=g\odot\frac{a-\mu}{\sqrt{\sigma^{2}+\varepsilon}}+b.$$

BatchNorm в RNN неудобен: статистики зависят от батча и от позиции $t$ (при переменных длинах — от числа «живых» примеров), поведение train/test расходится. LN не зависит ни от батча, ни от $T$, ставится на преактивации ворот, стабилизирует динамику состояния — и позже станет штатным блоком трансформера.

---

## LSTM

**LSTM** (long short-term memory) [(Hochreiter & Schmidhuber, 1997)](https://doi.org/10.1162/neco.1997.9.8.1735) разводит поток памяти и поток обработки. Вводится ячейка $c_t$, обновляемая аддитивно, — магистраль, вдоль которой ошибка распространяется назад без повторного умножения на $W$ и $f'$. В исходной архитектуре $c_t=c_{t-1}+i_t\odot\tilde c_t$, откуда $\partial c_t/\partial c_{t-1}=I$ — карусель постоянной ошибки (constant error carousel): градиент проходит сотни шагов не затухая. Скрытое состояние $h_t$ становится рабочей проекцией памяти, а не самой памятью.

Гейты (gates) — сигмоидные векторы из $(0,1)^d$, поэлементно умножающие информационные потоки: 0 — закрыто, 1 — открыто, между — частично. Степень пропускания вычисляется из текущего контекста $[h_{t-1};x_t]$: сеть сама решает, что помнить, что записывать и что показывать. Полная система:

$$\begin{aligned}
f_t&=\sigma(W_f[h_{t-1};x_t]+b_f) && \text{ворота забывания}\\
i_t&=\sigma(W_i[h_{t-1};x_t]+b_i) && \text{ворота входа}\\
\tilde c_t&=\tanh(W_c[h_{t-1};x_t]+b_c) && \text{кандидат}\\
c_t&=f_t\odot c_{t-1}+i_t\odot\tilde c_t && \text{обновление памяти}\\
o_t&=\sigma(W_o[h_{t-1};x_t]+b_o) && \text{ворота выхода}\\
h_t&=o_t\odot\tanh(c_t) && \text{выход}
\end{aligned}$$

### Gating

Ворот забывания в LSTM 1997 года не было — их добавили позже [(Gers et al., 2000)](https://doi.org/10.1162/089976600300015015), когда выяснилось, что на непрерывных потоках ячейка без механизма очистки дрейфует и насыщается. $f_t$ поэлементно масштабирует старую память: конец предложения — обнулить синтаксический контекст, смена темы — стереть тематический. Градиент магистрали теперь $\partial c_t/\partial c_{t-1}=\operatorname{diag}(f_t)$: скоростью «утечки» памяти управляет сама сеть и может держать её сколь угодно близко к единице

Запись разделена на «что» и «сколько»: кандидат $\tilde c_t\in(-1,1)^d$ предлагает содержимое, ворота $i_t$ дозируют запись. Это позволяет, например, полностью игнорировать шумный вход ($i_t\approx 0$), не трогая накопленное.

$h_t=o_t\odot\tanh(c_t)$: в памяти много служебного — счётчики, флаги вложенности, — и не всё это релевантно текущему предсказанию; $o_t$ фильтрует, что выставить наружу и передать дальше как $h_t$. tanh поверх $c_t$ возвращает ничем не ограниченную сверху магистраль в диапазон $(-1,1)$.

### Инициализация смещения ворот забывания

При случайной инициализации $f_t\approx 0.5$ — память умирает как $0.5^{T}$ ещё до начала обучения, и градиентный сигнал о пользе долгой памяти не успевает дойти до весов. Рецепт: $b_f=1\ldots 2$, тогда $\sigma(1)\approx 0.73$ — «по умолчанию помнить, забыванию — учиться». Масштабное сравнение тысяч вариантов архитектур [(Jozefowicz et al., 2015)](https://proceedings.mlr.press/v37/jozefowicz15.html) показало: LSTM с $b_f=1$ — трудно улучшаемый бейзлайн.

### Peephole-соединения

**Peephole-соединения** [(Gers & Schmidhuber, 2000)](https://doi.org/10.1109/IJCNN.2000.861302): ворота дополнительно «подглядывают» в ячейку, например $f_t=\sigma(W_f[h_{t-1};x_t]+p_f\odot c_{t-1}+b_f)$. Мотивация — точный тайминг: ворота видят накопленные счётчики напрямую, минуя фильтр $o_t$. Помогает в задачах точного счёта и ритма; в NLP систематического выигрыша не даёт и в стандартные реализации не вошло (см. абляции Greff et al. ниже).

---

## GRU

### Слияние ворот

**GRU** (gated recurrent unit) [(Cho et al., 2014)](https://arxiv.org/abs/1406.1078) — редукция LSTM до двух ворот без отдельной ячейки:

$$\begin{aligned}
z_t&=\sigma(W_z[h_{t-1};x_t])\\
r_t&=\sigma(W_r[h_{t-1};x_t])\\
\tilde h_t&=\tanh(W_h[r_t\odot h_{t-1};x_t])\\
h_t&=z_t\odot h_{t-1}+(1-z_t)\odot\tilde h_t
\end{aligned}$$

Ворота обновления $z_t$ выполняют работу пары forget+input со встроенной связью $f=1-i$: новое состояние — выпуклая комбинация старого и кандидата, при $z_t\to 1$ прошлое копируется, при $z_t\to 0$ замещается новым. «Раздуться», как $c_t$ у LSTM, оно не может; выходных ворот нет: $h_t$ — одновременно и память, и выход.

### Ворота сброса

$r_t$ действует до вычисления кандидата: $\tilde h_t$ может «не видеть» нерелевантное прошлое и начать с чистого листа, при этом само состояние ещё не стёрто — сотрёт его или нет, решит $z_t$. Разделение труда: $r$ — краткосрочная релевантность контекста, $z$ — долгосрочный баланс старого и нового.

### Параметрическая ёмкость

При входе размерности $e$ и состоянии $d$: LSTM — $4\big(d(d+e)+d\big)$ параметров (четыре блока), GRU — $3\big(d(d+e)+d\big)$ (три блока): на четверть меньше параметров и вычислений при том же $d$. У LSTM зато есть отдельный канал $c_t$ — $d$ измерений памяти, не проходящих через выходной фильтр.

### Что выбирать

Абляции [(Chung et al., 2014)](https://arxiv.org/abs/1412.3555): ворота дают колоссальный отрыв от vanilla RNN, а GRU и LSTM идут практически вровень. Поиск по пространству вариантов [(Greff et al., 2015)](https://arxiv.org/abs/1503.04069): критичны ворота забывания и выходная нелинейность, peephole и прочие вариации погоды не делают. Эвристика: мало данных или жёсткий бюджет — GRU (меньше параметров — меньше переобучение, быстрее); большие корпуса, языковое моделирование и перевод — LSTM (чуть выше потолок качества); универсальный дефолт — LSTM с $b_f=1$.

---

## Многослойные и двунаправленные архитектуры

### Стековые RNN

$$h_t^{(l)}=\operatorname{Cell}^{(l)}\big(h_t^{(l-1)},\,h_{t-1}^{(l)}\big)$$

— выходная последовательность слоя $l-1$ служит входной для слоя $l$. Глубина по слоям ортогональна глубине по времени и даёт иерархию представлений: нижние слои — орфография и морфология, верхние — синтаксис и семантика. Глубокие рекуррентные стеки впервые убедительно выстрелили в распознавании речи [(Graves et al., 2013)](https://arxiv.org/abs/1303.5778); в NLP типичны 2–4 слоя, глубже без остаточных связей качество не растёт.

### Двунаправленные RNN

**Bi-RNN** [(Schuster & Paliwal, 1997)](https://doi.org/10.1109/78.650093): две независимые сети читают последовательность слева направо и справа налево, представление шага — конкатенация $h_t=[\overrightarrow{h}_t;\overleftarrow{h}_t]$, а для классификации всей последовательности — $[\overrightarrow{h}_T;\overleftarrow{h}_1]$. Мотивация: у слова есть и левый, и правый контекст (омонимия часто разрешается словами справа). Стандарт для разметки, NER и энкодеров перевода. Кульминация направления — **ELMo** [(Peters et al., 2018)](https://arxiv.org/abs/1802.05365): предобученный многослойный biLSTM-LM как источник контекстных эмбеддингов, мост к эпохе pretrain–finetune.

### Ограничение для стриминга

Обратной сети нужен конец последовательности до начала работы: задержка равна длине входа. Онлайн-распознавание речи, синхронный перевод, автодополнение — только однонаправленные модели либо компромиссы вида блочной двунаправленности с ограниченным заглядыванием вперёд. Общий закон: полный правый контекст и реальное время несовместимы.

### Skip-соединения между слоями

Вертикальная глубина воспроизводит проблемы глубоких сетей — теперь по оси слоёв. Решение то же: остаточные связи $h^{(l)}=h^{(l-1)}+\operatorname{Cell}^{(l)}(\cdot)$ и прямые пробросы входа в верхние слои. Промышленный образец — **GNMT** [(Wu et al., 2016)](https://arxiv.org/abs/1609.08144), 8-слойный LSTM-стек с residual-связями.

---

## Encoder–Decoder (Seq2Seq)

### Схема

**Seq2Seq** (encoder–decoder) [(Cho et al., 2014)](https://arxiv.org/abs/1406.1078), [(Sutskever et al., 2014)](https://arxiv.org/abs/1409.3215): энкодер читает $x_{1..S}$ и сжимает её в контекстный вектор $v$ (обычно последнее состояние $h_S$); декодер — условная языковая модель $p(y_t\mid y_{<t},v)$, инициализированная $v$. Обучение — суммарная кросс-энтропия по цепному правилу. Инженерные находки Sutskever: четырёхслойные LSTM, разворот входной последовательности (первые слова источника оказываются рядом с первыми словами перевода — критические зависимости короче) и beam search при декодировании.

### Бутылочное горлышко

Смысл предложения любой длины протискивается через один вектор фиксированной размерности. Качество перевода заметно деградирует с ростом длины фразы [(Cho et al., 2014b)](https://arxiv.org/abs/1409.1259) — то самое информационное бутылочное горлышко. Наращивание размерности вектора не масштабируется; правильный ответ — дать декодеру доступ ко всем состояниям энкодера, то есть внимание.

### Teacher forcing

**Teacher forcing** [(Williams & Zipser, 1989)](https://doi.org/10.1162/neco.1989.1.2.270): при обучении вход декодера на шаге $t$ — золотой токен $y_{t-1}$, а не собственное предсказание. Это в точности максимизация правдоподобия факторизации $\prod_t p(y_t\mid y_{<t},x)$: обучение стабильно, ошибки не каскадируются, каждый шаг обусловлен корректным префиксом.

### Exposure bias

На инференсе золота нет — модель обусловлена собственными $\hat y_{<t}$, то есть распределением префиксов, которого она не видела при обучении. Это **exposure bias** [(Ranzato et al., 2015)](https://arxiv.org/abs/1511.06732): первая же ошибка уводит траекторию из носителя обучающего распределения, и ошибки накапливаются с длиной. Радикальные решения — обучение на уровне последовательностей (RL поверх метрики, beam-aware функции потерь), мягкое — scheduled sampling.

### Scheduled sampling

**Scheduled sampling** [(Bengio et al., 2015)](https://arxiv.org/abs/1506.03099): на каждом шаге обучения с вероятностью $\epsilon$ подаётся золотой токен, иначе — сэмпл самой модели; $\epsilon$ убывает по расписанию — линейному, экспоненциальному $\epsilon_i=k^{i}$ или обратно-сигмоидному $\epsilon_i=k/(k+e^{i/k})$ — от чистого teacher forcing к почти свободной генерации. Разрыв смягчается, хотя целевая функция перестаёт быть корректным правдоподобием (оценка смещена); на практике аккуратное расписание нередко помогает.

## Механизм внимания

**Механизм внимания** [(Bahdanau et al., 2014)](https://arxiv.org/abs/1409.0473) был предложен, чтобы устранить горлышко: декодер на каждом шаге генерации "видит" все токены входной последовательности и их представления полученные энкодером

1) оцениваем "сходство" текушего состояния декодера $s_t$ и представления каждого токена $h_s$ из входной послежовательности<br>$e_{t,s}=\operatorname{score}(s_{t-1},h_s)$<br><br>
2) все посчитанные сходства нормируем softmax, получаем вектор весов, суммируемый в 1:<br>
$\alpha_{t,s}=\frac{\exp e_{t,s}}{\sum_{s'}\exp e_{t,s'}}$<br><br>
3) считаем взвешенную сумму всех представлений энкодера<br>$c_t=\sum_{s=1}^{S}\alpha_{t,s}\,h_s$

Такой подход называют мягким выравниванием: вместо жёсткого выбора слов, как в статистическом переводе (IBM-модели), — соотвествие заменяется дифференцируемым распределением $\alpha_t$

<img src="img/attention1.png" width=250>

В оригинальном варианте Бахданау score считался аддитивно: $\operatorname{score}(s,h)=v_a^{\top}\tanh(W_a s+U_a h)$ — однослойная MLP, работает при разных размерностях, гибче при малых $d$. 

Вскоре [(Luong et al., 2015)](https://arxiv.org/abs/1508.04025) предложили мультипликативный способ учета связей: $s^{\top}h$ (скалярное произведение) или $s^{\top}W_a h$ (обощенное). Одно матричное умножение быстрее и проще; 

При больших размерностях модели $d$ скалярные произведения растут и softmax насыщается - отсюда позже масштабирование $q^{\top}k/\sqrt{d_k}$ в трансформере. Люонг также систематизировал global/local attention и input feeding — подачу $c_{t-1}$ на вход следующего шага.

### Контекстный вектор

$c_t$ — взвешенное среднее по всей входной последовательности называют контекстным вектором. Он пересобирается заново под каждый шаг выхода; конкатенируется с состоянием декодера перед предсказанием. Контекст перестал быть константой и стал функцией запроса. В терминах, которые скоро станут каноническими: $s$ — query, а $h_s$ — keys и values.

Матрицу весов $A=[\alpha_{t,s}]\in\mathbb{R}^{T_{output}\times T_{input}}$ называют Alignment матрицей. Её удобно использовать для интерпретации: тепловая карта показывает, куда «смотрело» каждое выходное слово. 

Для близких языков — почти диагональ; перестановки вида прилагательное–существительное en–fr видны изломами. Диагностика: равномерно размазанное или залипшее на одном токене внимание — типичный симптом недообученности либо ошибок маскирования.

### Градиентный поток

Внимание прокладывает от каждой потери $L_t$ к каждому состоянию энкодера $h_s$ путь длины $O(1)$ — в обход цепочки из десятков рекуррентных якобианов. Самый длинный и важный маршрут (выход → вход) спрямляется; затухание внутри самих цепочек RNN остаётся. Концептуально внимание — дифференцируемая адресация памяти по содержимому. Осталось заметить, что она справляется и без рекуррентной «несущей», — этот шаг сделает Transformer.

### Продвинутые архитектуры

**Pointer Networks** [(Vinyals et al., 2015)](https://arxiv.org/abs/1506.03134)<br>Энкодер-декодерная модель, которая на каждом шаге генерирует не новый токен, а позицию из входной последовательности - то есть как бы «указывает» пальцем на токены входа (отсюда название). Главное, что не требуется наличие словаря. Поэтому типовое применение - комбинаторные задачи (наполнение рюкзака, построение выпуклой оболочки), а также задачи экстрактивной суммаризации (основанной на выделении релевантных блоков текста). Архитектурно используется LSTM, позиция выбирается либо детерминированно через argmax, либо через случайное сэмплирование

**CopyNet** [(Gu et al., 2016)](https://arxiv.org/abs/1603.06393)<br>
Разрешим модели иметь свой небольшой словарь и пусть на каждом шаге генерации она может выбрать токен из словаря или токен из входа. Как выбирается - складываем два скора, сортируем сумму по убыванию, применяем softmax для нормировки и сэмплируем. 

Вероятность генерации токена из словаря реализуется стандартно, через линейный слой $Wx+b$. Вероятность выбора токена $w_i$ моделируется линейным навесом над конкатенацией $[s_j, c_j, h_i]$, где s_j  c_j h_i Решает проблему OOV-имён и редких сущностей в суммаризации и диалоге

**Pointer-generator** [(See et al., 2017)](https://arxiv.org/abs/1704.04368):<br>
Та же идея, но две вероятности замешиваются с разными весами

- вероятность выбора токена $w$ из словаря считаем стандартно для seq2seq: $P_{\text{vocab}}(w) = \text{softmax}\left( \mathbf{W}_g \mathbf{s}_t + \mathbf{b}_g \right)$
- вероятность выбора токена из входной последовательности считаем, просто как его attention вес: $P_{\text{copy}}(w) = \sum_{j: x_j = w} \alpha_{t,j}$
- коэффициент замешивания двух сигналов считаем линейным слоем по всем доступным данным: $p_{\text{gen}} = \sigma\left( \mathbf{W}_p [\mathbf{s}_t; \mathbf{c}_t; \mathbf{e}(y_{t-1})] + \mathbf{b}_p \right)$

**Neural Turing Machine** [(Graves et al., 2014)](https://arxiv.org/abs/1410.5401): контроллер-LSTM плюс внешняя матрица памяти $M\in\mathbb{R}^{N\times M}$ с дифференцируемыми чтением $r_t=\sum_i w_t(i)\,M_t(i)$ и записью; адресация контентная (косинусная близость с softmax) и позиционная (сдвиги, интерполяция). Обучается алгоритмам копирования и сортировки по примерам вход–выход. 

**Differentiable Neural Computer** [(Graves et al., 2016)](https://doi.org/10.1038/nature20101) добавляет динамическую аллокацию и темпоральные связи между записями, решает графовые задачи. Концептуальный вклад — разделение вычислителя и памяти и адресация по содержимому: дальние предки современных retrieval-механизмов

**Stack-Augmented RNN** [(Joulin & Mikolov, 2015)](https://arxiv.org/abs/1503.01007): дифференцируемый стек с мягкими push/pop. RNN конечной точности — по существу конечный автомат; стек поднимает модель на ступень выше по иерархии Хомского, давая контекстно-свободные способности: $a^{n}b^{n}$, вложенные скобки, счётчики — то, что нужно синтаксису с неограниченной вложенностью.

## Практические кейсы

### Char-RNN: генерация текста посимвольно

**Char-RNN** [(Karpathy, 2015)](http://karpathy.github.io/2015/05/21/rnn-effectiveness/): словарь — символы (порядка 50–100), два-три слоя LSTM, softmax по алфавиту; обучение — предсказание следующего символа с truncated BPTT; генерация — авторегрессивное сэмплирование с температурой $p_i\propto\exp(z_i/\tau)$: $\tau<1$ — консервативнее, $\tau>1$ — разнообразнее. Без токенизатора модель выучивает орфографию, пунктуацию, парность скобок, структуру кода и LaTeX — наглядное свидетельство того, что предсказание следующего токена вынуждает усваивать структуру языка. Прямой идейный предок GPT.

### Классификация тональности

Пайплайн: токенизация → матрица эмбеддингов, инициализированная **Word2Vec** [(Mikolov et al., 2013)](https://arxiv.org/abs/1301.3781) или **GloVe** [(Pennington et al., 2014)](https://aclanthology.org/D14-1162/) (замораживать при малых данных, дообучать при больших) → biLSTM → агрегация по времени (последнее состояние либо mean/max-пулинг, строго с маской) → полносвязный слой. Типичные ошибки: пулинг по PAD-позициям и токенизация, не совпадающая со словарём эмбеддингов; против переобучения — вариационный dropout и ранняя остановка.

### Прогнозирование временных рядов

Многомерный ряд режется скользящими окнами: вход $T_{вх}$ шагов, цель — горизонт $H$. Нормализация (z-score по каналам) только по train-статистикам; валидация — строго по времени, никакого случайного сплита. Стратегии multi-step: direct ($H$ выходов сразу) и iterative (авторегрессивно; накапливает ошибку — тот же exposure bias в новом обличье). Не NLP, но канонический полигон many-to-many режима RNN.

### Машинный перевод как сборка главы

Полный проект: сабвордная токенизация **BPE** [(Sennrich et al., 2016)](https://arxiv.org/abs/1508.07909) → biLSTM-энкодер + LSTM-декодер + внимание Луонга → teacher forcing, опционально scheduled sampling → beam search (ширина 4–10, нормализация по длине) → метрика **BLEU** [(Papineni et al., 2002)](https://aclanthology.org/P02-1040/). Чек-лист из пройденного: packing, клиппинг по норме, $b_f=1$, вариационный dropout, LayerNorm, контроль матрицы внимания.

## Интерпретация и отладка

### Проекции скрытых состояний

Собрать $h_t$ по корпусу и спроецировать в 2D: **t-SNE** [(van der Maaten & Hinton, 2008)](https://jmlr.org/papers/v9/vandermaaten08a.html) или **UMAP** [(McInnes et al., 2018)](https://arxiv.org/abs/1802.03426), раскрасив точки метками (часть речи, тональность, язык). Сложившиеся кластеры показывают, что именно закодировано; траектория состояний одного документа — как дрейфует контекст во времени.

### Активации ворот

Гистограммы $f_t$, $i_t$, $z_t$ по датасету: масса у единицы — долгая память, у нуля — постоянные сбросы, бимодальность — здоровая специализация нейронов. Классический анализ [(Karpathy et al., 2015)](https://arxiv.org/abs/1506.02078) нашёл интерпретируемые ячейки: детектор «внутри кавычек», счётчик длины строки, датчик вложенности скобок — память LSTM реально используется по назначению.

### Мониторинг градиентов

Логировать $\lVert\partial L/\partial h_t\rVert$ как функцию расстояния от места потерь: экспоненциальный спад — картина затухания. Частота срабатывания клиппинга — отдельная метрика: если порог бьётся на каждом шаге, $\theta$ слишком мал или learning rate велик. Гистограммы норм градиентов по слоям — вертикальная диагностика стека.

### Санити-чеки

Модель обязана идеально переобучиться на крошечном срезе — одном батче, коротких подпоследовательностях; если лосс не идёт к нулю, ищите баг (маски, packing, случайный detach, learning rate), а не «сложность задачи». Обратные проверки: перестановка паддинга не должна менять выход (корректность маскирования), а перемешивание меток должно ломать обучение (нет утечки).

## Критика RNN и эволюция подходов

### Отсутствие параллелизма по времени

Фундаментальное ограничение: $h_t$ невычислим раньше $h_{t-1}$ — $O(T)$ последовательных операций на обучении, конвейеры GPU/TPU простаивают, длина контекста упирается не в память, а во время. Именно это, а не качество, решило исход конкуренции архитектур.

### Сравнение с self-attention

**Transformer** [(Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762) делает радикальный вывод из успеха внимания: рекуррентность не нужна. Self-attention связывает любые две позиции путём длины $O(1)$ и считается параллельно по всей оси времени; порядок возвращается позиционными кодировками.

| Механизм | Вычислений на слой | Последовательных операций | Длина пути градиента |
|---|---|---|---|
| Рекуррентный | $O(T\,d^{2})$ | $O(T)$ | $O(T)$ |
| Self-attention | $O(T^{2}\,d)$ | $O(1)$ | $O(1)$ |
| Свёрточный (ядро $k$) | $O(k\,T\,d^{2})$ | $O(1)$ | $O(\log_k T)$ |

Цена — квадратичность по длине и растущий KV-кэш на инференсе. RNN проиграли не качеством на своих масштабах, а масштабируемостью: параллелизм обучения позволил влить на порядки больше данных и параметров — началась эпоха BERT и GPT.

### Свёрточная альтернатива

**TCN** [(Bai et al., 2018)](https://arxiv.org/abs/1803.01271) развивает каузальные дилатированные свёртки **WaveNet** [(van den Oord et al., 2016)](https://arxiv.org/abs/1609.03499): рецептивное поле растёт экспоненциально с глубиной, обучение параллельно по $T$; на ряде последовательностных бенчмарков TCN обходит LSTM и GRU. Ограничение — память жёстко ограничена рецептивным полем, а состояние для стриминга — буфер длиной в это поле, не компактный вектор.

### Возвращение рекуррентности: SSM

**S4** [(Gu et al., 2021)](https://arxiv.org/abs/2111.00396): линейная рекурсия $h_t=\bar A h_{t-1}+\bar B x_t$, $y_t=C h_t$ (дискретизация непрерывной state-space системы). Отсутствие нелинейности между шагами даёт двойственность: обучение — свёртка с ядром $\bar K=(C\bar B,\,C\bar A\bar B,\,C\bar A^{2}\bar B,\dots)$ через FFT, параллельно; инференс — рекуррентно за $O(1)$ на шаг. Долгая память достигается не воротами, а специальной инициализацией $A$ (HiPPO); прорыв на Long Range Arena, где трансформеры проваливались. **Mamba** [(Gu & Dao, 2023)](https://arxiv.org/abs/2312.00752) добавляет селективность: $\bar B$, $C$ и шаг дискретизации $\Delta$ становятся функциями входа — контентно-зависимая фильтрация, функциональный наследник ворот LSTM; свёрточная форма теряется, вместо неё аппаратно-оптимизированный параллельный scan. Линейное время, константное состояние на инференсе, качество на уровне трансформеров при сопоставимых бюджетах. Параллельная линия — линейное внимание как RNN [(Katharopoulos et al., 2020)](https://arxiv.org/abs/2006.16236), **RWKV** [(Peng et al., 2023)](https://arxiv.org/abs/2305.13048) и **xLSTM** [(Beck et al., 2024)](https://arxiv.org/abs/2405.04517) с экспоненциальными воротами и матричной памятью. Круг замкнулся: рекуррентное состояние и ворота вернулись в мейнстрим как ответ на квадратичность внимания.

### Место классических RNN сегодня

Нишу определяет свойство $O(1)$-состояния: стриминг с жёсткой латентностью (онлайн-распознавание речи на **RNN-T** [(Graves, 2012)](https://arxiv.org/abs/1211.3711) годами работало в продакшене), edge-устройства и микроконтроллеры, малые датасеты и короткие последовательности (LSTM — по-прежнему сильный бейзлайн против переобучающегося трансформера), временные ряды и сенсорика. Плюс дидактика: состояние, ворота, teacher forcing, exposure bias — понятия, введённые здесь, работают во всём современном стеке.

## Инженерная оптимизация и развёртывание

### Gradient checkpointing

**Gradient checkpointing** [(Chen et al., 2016)](https://arxiv.org/abs/1604.06174): хранить активации только в контрольных точках (например, каждые $\sqrt{T}$ шагов), промежуточные пересчитывать на обратном проходе: память $O(\sqrt{T})$ вместо $O(T)$ ценой примерно +33% вычислений (один дополнительный форвард). Для RNN контрольные точки естественно ставятся вдоль оси времени; в отличие от truncated BPTT градиент не обрезается — считается честно.

### Смешанная точность

**Mixed precision** [(Micikevicius et al., 2017)](https://arxiv.org/abs/1710.03740): вычисления в fp16/bf16, мастер-копия весов в fp32, loss scaling против underflow градиентов. Специфика RNN: тысячи последовательных шагов накапливают ошибку округления, а нормы состояний гуляют широко — узкая экспонента fp16 чревата переполнениями, bf16 надёжнее; и стоит использовать фьюзнутые cuDNN-ядра, иначе выигрыш съедается запуском множества мелких ядер.

### Квантизация для edge

INT8-квантизация весов и активаций даёт около четырёхкратной экономии памяти и заметное ускорение на CPU/NPU. Тонкость рекуррентности: ошибка квантизации состояния проходит через одну и ту же ячейку многократно и накапливается по шагам — на длинных последовательностях post-training квантизация деградирует, предпочтительнее quantization-aware training; аккумуляторы держать в int32, масштабы весов — поканальные, сигмоиды и tanh — таблицами.

### Экспорт в ONNX

Экспортировать с динамическими осями: `dynamic_axes={'x': {0: 'batch', 1: 'time'}}` — иначе граф зафиксирует длины, встреченные при трассировке. LSTM/GRU отображаются во фьюзнутые операторы ONNX (кастомные ячейки — в циклы Scan/Loop, заметно медленнее). Рантаймы (ONNX Runtime, TensorRT) фьюзят операции и планируют статический граф; обязательная проверка — численный паритет с исходной моделью на батчах переменной длины: главный источник расхождений — маски и packing.

## Итоги

Арка главы — арка одной задачи: как протащить информацию вперёд, а градиент назад сквозь длинную последовательность. Vanilla RNN дала форму (состояние плюс разделение весов во времени), но степень якобиана убила дальние зависимости. LSTM и GRU ответили аддитивной памятью и воротами; глубина и двунаправленность добавили ёмкости; seq2seq превратил RNN в универсальный преобразователь последовательностей, упёрся в бутылочное горлышко — и породил внимание. Внимание оказалось сильнее носителя: Transformer выбросил рекуррентность ради параллелизма обучения, а спустя пять лет state-space модели вернули рекуррентное состояние, чтобы победить квадратичность внимания. Следующая тема начинается ровно с вопроса, которым заканчивается эта: что, если внимание — это всё, что нужно?

| Годы | Веха | Работы |
|---|---|---|
| 1990 | Простая рекуррентная сеть со скрытым состоянием; BPTT | Elman; Werbos |
| 1991–1994 | Диагноз затухающих и взрывающихся градиентов | Hochreiter; Bengio et al. |
| 1997 | LSTM (карусель постоянной ошибки); Bi-RNN | Hochreiter, Schmidhuber; Schuster, Paliwal |
| 2000 | Ворота забывания; peephole-соединения | Gers et al. |
| 2010–2013 | Рекуррентные языковые модели; глубокие RNN в речи; анализ градиентов и клиппинг | Mikolov et al.; Graves et al.; Pascanu et al. |
| 2013–2014 | Word2Vec, GloVe; GRU; Seq2Seq | Mikolov et al.; Cho et al.; Sutskever et al. |
| 2014–2015 | Внимание (Бахданау, Луонг); NTM; Pointer Networks; scheduled sampling | Bahdanau et al.; Graves et al.; Vinyals et al.; Bengio et al. |
| 2015–2016 | Вариационный dropout; zoneout; LayerNorm; GNMT | Gal, Ghahramani; Krueger et al.; Ba et al.; Wu et al. |
| 2017 | Transformer: отказ от рекуррентности | Vaswani et al. |
| 2018 | ELMo; TCN | Peters et al.; Bai et al. |
| 2020–2024 | Возврат рекуррентности: линейное внимание, S4, RWKV, Mamba, xLSTM | Katharopoulos et al.; Gu et al.; Peng et al.; Gu, Dao; Beck et al. |